# EZHit Fine-tuning Colab

This notebook fine-tunes an EZHit / HybridPairKAN enzyme–reaction compatibility model on a user-provided dataset.

## What you need to upload

1. A pretrained EZHit checkpoint, for example `binarycls_best_val_seed40.pt`.
2. A CSV dataset with at least these columns:

| column | meaning |
|---|---|
| `protein_sequence` | enzyme amino-acid sequence |
| `CANO_RXN_SMILES` | reaction SMILES in `reactants>>products` format |
| `Label` | binary label, `1` for compatible enzyme–reaction pair and `0` for negative pair |

Optional columns:

| column | meaning |
|---|---|
| `split` | `train`, `val`, or `test`; if absent, the notebook will create group-aware splits |
| `group_id` | group key for ranking; if absent, reaction SMILES will be used |

Outputs:

- fine-tuned checkpoint
- feature caches
- test predictions
- `train_distribution_stat.pt` for Mahalanobis-distance filtering in the HuggingFace Space


In [ ]:
# @title 1. Install dependencies
# Colab runtime: GPU is strongly recommended.
# If installation fails for pykan in your environment, restart the runtime and rerun this cell.

!pip -q install pykan drfp rdkit-pypi transformers sentencepiece accelerate scikit-learn pandas tqdm


In [ ]:
# @title 2. Upload pretrained checkpoint and dataset CSV
from google.colab import files
import os, shutil, glob

os.makedirs("input", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("cache", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

uploaded = files.upload()

for name in uploaded.keys():
    if name.endswith(".csv"):
        shutil.move(name, f"input/{name}")
    elif name.endswith(".pt") or name.endswith(".pth"):
        shutil.move(name, f"checkpoints/{name}")
    else:
        shutil.move(name, f"input/{name}")

print("CSV files:", glob.glob("input/*.csv"))
print("Checkpoint files:", glob.glob("checkpoints/*.pt") + glob.glob("checkpoints/*.pth"))


In [ ]:
# @title 3. Configuration

import glob
import os

# Required files.
DATA_CSV = glob.glob("input/*.csv")[0]
PRETRAINED_CKPT = (glob.glob("checkpoints/*.pt") + glob.glob("checkpoints/*.pth"))[0]

# Column names.
SEQ_COL = "protein_sequence"
RXN_COL = "CANO_RXN_SMILES"
LABEL_COL = "Label"
GROUP_COL = "CANO_RXN_SMILES"   # used for ranking metrics and group-aware split

# Model / feature dimensions.
PROTT5_DIM = 1024
FP_DIM = 2048
REACT_DIM = 2048

# Training parameters.
SEED = 42
EPOCHS = 15
BATCH_SIZE = 128      # increase to 256/512 if GPU memory is enough
LR = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
PATIENCE = 3
MIN_DELTA = 0.002
SELECT_METRIC = "top10"  # choices: top1, top10, dcg@10, auc_micro, auprc_micro
KAN_REG_WEIGHT = 1e-4
POS_WEIGHT = "auto"      # "auto" or a float, e.g. 5.0

# Small-data option.
# If your dataset is very small, setting this True can reduce overfitting.
FREEZE_COMPRESSOR = False

# Mahalanobis statistic.
# This will be generated from positive training pairs after fine-tuning.
MAHA_EPS = 1e-4

print("DATA_CSV:", DATA_CSV)
print("PRETRAINED_CKPT:", PRETRAINED_CKPT)


In [ ]:
# @title 4. Imports and utility functions

import os
import re
import math
import random
from typing import Any, Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

from kan import KAN

from drfp import DrfpEncoder

from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
RDLogger.DisableLog("rdApp.*")

from transformers import T5Tokenizer, T5EncoderModel

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def l2norm_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(n, eps)

def validate_dataset(df: pd.DataFrame):
    required = [SEQ_COL, RXN_COL, LABEL_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Current columns: {list(df.columns)}")
    df = df.copy()
    df[SEQ_COL] = df[SEQ_COL].fillna("").astype(str).str.strip()
    df[RXN_COL] = df[RXN_COL].fillna("").astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    df = df[(df[SEQ_COL] != "") & (df[RXN_COL].str.contains(">>", regex=False))]
    df = df[df[LABEL_COL].isin([0, 1])]
    df = df.reset_index(drop=True)
    if len(df) == 0:
        raise ValueError("No valid rows after filtering. Please check sequences, reaction SMILES, and labels.")
    return df

def make_group_splits(df: pd.DataFrame, group_col: str, seed: int = 42):
    if "split" in df.columns:
        split = df["split"].astype(str).str.lower()
        df_tr = df[split == "train"].reset_index(drop=True)
        df_va = df[split.isin(["val", "valid", "validation"])].reset_index(drop=True)
        df_te = df[split == "test"].reset_index(drop=True)
        if len(df_tr) > 0 and len(df_va) > 0:
            return df_tr, df_va, df_te

    group_values = df[group_col].astype(str).values if group_col in df.columns else df[RXN_COL].astype(str).values
    n_groups = len(set(group_values))

    if n_groups >= 5:
        gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
        tr_idx, temp_idx = next(gss1.split(df, df[LABEL_COL], groups=group_values))
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_temp = df.iloc[temp_idx].reset_index(drop=True)

        temp_groups = df_temp[group_col].astype(str).values if group_col in df_temp.columns else df_temp[RXN_COL].astype(str).values
        if len(set(temp_groups)) >= 2:
            gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
            va_idx, te_idx = next(gss2.split(df_temp, df_temp[LABEL_COL], groups=temp_groups))
            df_va = df_temp.iloc[va_idx].reset_index(drop=True)
            df_te = df_temp.iloc[te_idx].reset_index(drop=True)
        else:
            df_va = df_temp
            df_te = df_temp.iloc[0:0].copy()
    else:
        train_df, temp_df = train_test_split(
            df, test_size=0.20, random_state=seed, stratify=df[LABEL_COL] if df[LABEL_COL].nunique() > 1 else None
        )
        if len(temp_df) >= 2:
            df_va, df_te = train_test_split(
                temp_df, test_size=0.50, random_state=seed,
                stratify=temp_df[LABEL_COL] if temp_df[LABEL_COL].nunique() > 1 and temp_df[LABEL_COL].value_counts().min() >= 2 else None
            )
        else:
            df_va = temp_df
            df_te = temp_df.iloc[0:0].copy()
        df_tr = train_df.reset_index(drop=True)
        df_va = df_va.reset_index(drop=True)
        df_te = df_te.reset_index(drop=True)

    return df_tr, df_va, df_te


In [ ]:
# @title 5. Load CSV and create splits

df = pd.read_csv(DATA_CSV, low_memory=False)
df = validate_dataset(df)

if GROUP_COL not in df.columns:
    print(f"GROUP_COL '{GROUP_COL}' not found. Using reaction SMILES as the group key.")
    GROUP_COL = RXN_COL

df_tr, df_va, df_te = make_group_splits(df, GROUP_COL, seed=SEED)

print("Rows:")
print("  train:", len(df_tr), "pos:", int(df_tr[LABEL_COL].sum()), "neg:", int((df_tr[LABEL_COL] == 0).sum()))
print("  val:  ", len(df_va), "pos:", int(df_va[LABEL_COL].sum()), "neg:", int((df_va[LABEL_COL] == 0).sum()))
print("  test: ", len(df_te), "pos:", int(df_te[LABEL_COL].sum()), "neg:", int((df_te[LABEL_COL] == 0).sum()))

df_tr.to_csv("outputs/train_split.csv", index=False)
df_va.to_csv("outputs/val_split.csv", index=False)
df_te.to_csv("outputs/test_split.csv", index=False)


In [ ]:
# @title 6. Build or load ProtT5 embeddings

def preprocess_seq_for_prott5(seq: str, max_len: int = 1022) -> str:
    seq = seq[:max_len]
    seq = re.sub(r"[UZOBuzob]", "X", seq.upper())
    return " ".join(list(seq))

def mean_pool_reps(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask_expanded = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask_expanded).sum(dim=1)
    denom = mask_expanded.sum(dim=1).clamp(min=1e-9)
    return summed / denom

@torch.no_grad()
def embed_sequences_prott5(
    sequences: List[str],
    cache_path: str,
    batch_size: int = 4,
    max_len: int = 1022
) -> torch.Tensor:
    if os.path.exists(cache_path):
        obj = torch.load(cache_path, map_location="cpu")
        if torch.is_tensor(obj) and obj.ndim == 2 and obj.shape[0] == len(sequences):
            print(f"[Cache] loaded {cache_path} {tuple(obj.shape)}")
            return obj.float()
        print(f"[Cache] ignored invalid cache: {cache_path}")

    model_name = "Rostlab/prot_t5_xl_half_uniref50-enc"
    tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)

    if torch.cuda.is_available():
        encoder = T5EncoderModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        ).to(device)
    else:
        encoder = T5EncoderModel.from_pretrained(model_name).to(device)

    encoder.eval()
    outputs_all = []

    for i in tqdm(range(0, len(sequences), batch_size), desc=f"ProtT5 -> {os.path.basename(cache_path)}"):
        batch = [preprocess_seq_for_prott5(s, max_len=max_len) for s in sequences[i:i+batch_size]]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len + 1
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        if torch.cuda.is_available():
            with torch.autocast("cuda", dtype=torch.float16):
                out = encoder(**inputs)
        else:
            out = encoder(**inputs)

        pooled = mean_pool_reps(out.last_hidden_state, inputs["attention_mask"])
        outputs_all.append(pooled.detach().cpu().float())

    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    emb = torch.cat(outputs_all, dim=0).float()
    torch.save(emb, cache_path)
    print(f"[Cache] saved {cache_path} {tuple(emb.shape)}")
    return emb

esm_tr = embed_sequences_prott5(df_tr[SEQ_COL].tolist(), "cache/train_prott5.pt")
esm_va = embed_sequences_prott5(df_va[SEQ_COL].tolist(), "cache/val_prott5.pt")
esm_te = embed_sequences_prott5(df_te[SEQ_COL].tolist(), "cache/test_prott5.pt") if len(df_te) > 0 else None

print("Embedding shapes:", esm_tr.shape, esm_va.shape, None if esm_te is None else esm_te.shape)


In [ ]:
# @title 7. Build or load reaction features

def drfp_encode(smiles_list: List[str], n_folded_length: int = 2048) -> np.ndarray:
    fps = DrfpEncoder.encode(smiles_list, n_folded_length=n_folded_length)
    return np.asarray(fps, dtype=np.float32)

def make_drfp_rowaligned_dedup(df: pd.DataFrame, rxn_col: str, fp_dim: int, cache_path: str) -> torch.Tensor:
    if os.path.exists(cache_path):
        obj = torch.load(cache_path, map_location="cpu")
        if torch.is_tensor(obj) and obj.shape == (len(df), fp_dim):
            print(f"[Cache] loaded {cache_path} {tuple(obj.shape)}")
            return obj.float()

    smis = df[rxn_col].fillna("").astype(str).tolist()
    uniq, smi2uid = [], {}
    uids = np.empty((len(smis),), dtype=np.int64)

    for i, s in enumerate(smis):
        uid = smi2uid.get(s)
        if uid is None:
            uid = len(uniq)
            smi2uid[s] = uid
            uniq.append(s)
        uids[i] = uid

    fps_u = np.zeros((len(uniq), fp_dim), dtype=np.float32)
    chunk = 2048
    for i in tqdm(range(0, len(uniq), chunk), desc=f"DRFP -> {os.path.basename(cache_path)}"):
        part = uniq[i:i+chunk]
        fps_u[i:i+len(part)] = drfp_encode(part, n_folded_length=fp_dim)

    fps_u = l2norm_np(fps_u)
    fps = torch.tensor(fps_u[uids], dtype=torch.float32)
    torch.save(fps, cache_path)
    print(f"[Cache] saved {cache_path} {tuple(fps.shape)}")
    return fps

def get_reactant_morgan_fp(rxn_smiles: str, radius: int = 2, nBits: int = 2048) -> np.ndarray:
    if not isinstance(rxn_smiles, str) or ">>" not in rxn_smiles:
        return np.zeros((nBits,), dtype=np.float32)

    reactants_smi = rxn_smiles.split(">>")[0]
    mol = Chem.MolFromSmiles(reactants_smi)
    if mol is None:
        return np.zeros((nBits,), dtype=np.float32)

    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.float32)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def make_reactant_rowaligned_dedup(df: pd.DataFrame, rxn_col: str, fp_dim: int, cache_path: str) -> torch.Tensor:
    if os.path.exists(cache_path):
        obj = torch.load(cache_path, map_location="cpu")
        if torch.is_tensor(obj) and obj.shape == (len(df), fp_dim):
            print(f"[Cache] loaded {cache_path} {tuple(obj.shape)}")
            return obj.float()

    smis = df[rxn_col].fillna("").astype(str).tolist()
    uniq, smi2uid = [], {}
    uids = np.empty((len(smis),), dtype=np.int64)

    for i, s in enumerate(smis):
        uid = smi2uid.get(s)
        if uid is None:
            uid = len(uniq)
            smi2uid[s] = uid
            uniq.append(s)
        uids[i] = uid

    fps_u = np.zeros((len(uniq), fp_dim), dtype=np.float32)
    for i, s in enumerate(tqdm(uniq, desc=f"Reactant Morgan -> {os.path.basename(cache_path)}")):
        fps_u[i] = get_reactant_morgan_fp(s, radius=2, nBits=fp_dim)

    fps_u = l2norm_np(fps_u)
    fps = torch.tensor(fps_u[uids], dtype=torch.float32)
    torch.save(fps, cache_path)
    print(f"[Cache] saved {cache_path} {tuple(fps.shape)}")
    return fps

fp_tr = make_drfp_rowaligned_dedup(df_tr, RXN_COL, FP_DIM, "cache/train_drfp.pt")
fp_va = make_drfp_rowaligned_dedup(df_va, RXN_COL, FP_DIM, "cache/val_drfp.pt")
fp_te = make_drfp_rowaligned_dedup(df_te, RXN_COL, FP_DIM, "cache/test_drfp.pt") if len(df_te) > 0 else None

react_tr = make_reactant_rowaligned_dedup(df_tr, RXN_COL, REACT_DIM, "cache/train_reactant.pt")
react_va = make_reactant_rowaligned_dedup(df_va, RXN_COL, REACT_DIM, "cache/val_reactant.pt")
react_te = make_reactant_rowaligned_dedup(df_te, RXN_COL, REACT_DIM, "cache/test_reactant.pt") if len(df_te) > 0 else None


In [ ]:
# @title 8. Model, metrics, and helper functions

class HybridPairKAN(nn.Module):
    def __init__(self, esm_dim: int, drfp_dim: int, react_dim: int, hidden: int, dropout: float = 0.0):
        super().__init__()
        self.esm_ln = nn.LayerNorm(esm_dim)
        self.drfp_ln = nn.LayerNorm(drfp_dim)
        self.react_ln = nn.LayerNorm(react_dim)

        in_dim = esm_dim + drfp_dim + react_dim
        self.bottleneck_dim = hidden
        self.compressor = nn.Sequential(
            nn.Linear(in_dim, self.bottleneck_dim),
            nn.LayerNorm(self.bottleneck_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.net = KAN([self.bottleneck_dim, hidden, 1])

    def forward(
        self,
        esm: torch.Tensor,
        drfp: torch.Tensor,
        react: torch.Tensor,
        update_grid: bool = False,
        return_latent: bool = False
    ):
        e_norm = self.esm_ln(esm)
        d_norm = self.drfp_ln(drfp)
        r_norm = self.react_ln(react)
        x = torch.cat([e_norm, d_norm, r_norm], dim=-1)
        latent = self.compressor(x)
        logit = self.net(latent, update_grid=update_grid).squeeze(-1)
        if return_latent:
            return logit, latent
        return logit

def dcg_at_k_binary(labels_sorted: np.ndarray, k: int = 10) -> float:
    k = min(k, len(labels_sorted))
    if k <= 0:
        return 0.0
    rel = labels_sorted[:k].astype(np.float32)
    denom = np.log2(np.arange(2, k + 2, dtype=np.float32))
    return float((rel / denom).sum())

def topk_success(labels_sorted: np.ndarray, k: int) -> float:
    k = min(k, len(labels_sorted))
    if k <= 0:
        return 0.0
    return 1.0 if labels_sorted[:k].sum() > 0 else 0.0

@torch.no_grad()
def predict_scores_and_latents(
    model: nn.Module,
    esm: torch.Tensor,
    fp: torch.Tensor,
    react: torch.Tensor,
    batch_size: int = 4096,
    return_latents: bool = False
):
    model.eval()
    scores = np.zeros((esm.shape[0],), dtype=np.float32)
    latents = [] if return_latents else None

    for i in range(0, esm.shape[0], batch_size):
        e = esm[i:i+batch_size].to(device)
        f = fp[i:i+batch_size].to(device)
        r = react[i:i+batch_size].to(device)

        if return_latents:
            logit, z = model(e, f, r, return_latent=True)
            latents.append(z.detach().cpu().float())
        else:
            logit = model(e, f, r)

        prob = torch.sigmoid(logit).detach().cpu().numpy().astype(np.float32)
        scores[i:i+len(prob)] = prob

    if return_latents:
        return scores, torch.cat(latents, dim=0)
    return scores

def evaluate_groups_ranking(df: pd.DataFrame, scores: np.ndarray, group_key: str, label_col: str) -> Dict[str, float]:
    labels_all = df[label_col].to_numpy(dtype=np.int32)

    if labels_all.min() != labels_all.max():
        auc_micro = float(roc_auc_score(labels_all, scores))
        auprc = float(average_precision_score(labels_all, scores))
    else:
        auc_micro, auprc = float("nan"), float("nan")

    groups = df.groupby(group_key, sort=False).indices
    topKs = [1, 3, 5, 10]
    topk_hits = {k: 0.0 for k in topKs}
    dcg10_sum, valid_groups = 0.0, 0

    for key, idxs_raw in groups.items():
        idxs = np.array(idxs_raw, dtype=np.int64)
        lab = labels_all[idxs]
        sc = scores[idxs]

        if lab.sum() <= 0:
            continue

        valid_groups += 1
        lab_sorted = lab[np.argsort(-sc)]

        for kk in topKs:
            topk_hits[kk] += topk_success(lab_sorted, kk)

        dcg10_sum += dcg_at_k_binary(lab_sorted, 10)

    if valid_groups == 0:
        return {
            "top1": 0.0, "top3": 0.0, "top5": 0.0, "top10": 0.0,
            "dcg@10": 0.0, "auc_micro": auc_micro, "auprc_micro": auprc,
            "valid_groups": 0.0, "total_groups": float(len(groups))
        }

    out = {f"top{kk}": float(topk_hits[kk] / valid_groups) for kk in topKs}
    out.update({
        "dcg@10": float(dcg10_sum / valid_groups),
        "auc_micro": auc_micro,
        "auprc_micro": auprc,
        "valid_groups": float(valid_groups),
        "total_groups": float(len(groups))
    })
    return out

def calculate_local_ef(
    df: pd.DataFrame,
    scores: np.ndarray,
    group_key: str,
    label_col: str,
    fractions: List[float] = [0.01, 0.02, 0.05, 0.1]
) -> Dict[str, float]:
    tmp = df.copy()
    tmp["_score"] = scores
    tmp["_label"] = tmp[label_col].astype(int)

    ef_lists = {f: [] for f in fractions}
    valid_groups = 0

    for _, grp in tmp.groupby(group_key):
        y = grp["_label"].values
        s = grp["_score"].values
        n = len(y)
        n_pos = y.sum()

        if n_pos == 0:
            continue

        sorted_labels = y[np.argsort(-s)]

        for frac in fractions:
            k = min(max(int(n * frac), 5), n)
            hits = sorted_labels[:k].sum()
            random_expectation = n_pos * frac
            ef_lists[frac].append(0.0 if random_expectation == 0 else hits / random_expectation)

        valid_groups += 1

    metrics = {f"ef@{frac}": float(np.mean(values)) if values else 0.0 for frac, values in ef_lists.items()}
    metrics["ef_valid_groups"] = float(valid_groups)
    return metrics


In [ ]:
# @title 9. Load pretrained checkpoint

ckpt = torch.load(PRETRAINED_CKPT, map_location="cpu")
ckpt_args = ckpt.get("args", {})

hidden_dim = ckpt_args.get("hidden", 512)

model = HybridPairKAN(
    esm_dim=esm_tr.shape[1],
    drfp_dim=fp_tr.shape[1],
    react_dim=react_tr.shape[1],
    hidden=hidden_dim,
    dropout=DROPOUT
).to(device)

state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
missing, unexpected = model.load_state_dict(state, strict=False)

print("Loaded pretrained checkpoint.")
print("hidden_dim:", hidden_dim)
print("missing keys:", missing)
print("unexpected keys:", unexpected)

if FREEZE_COMPRESSOR:
    for p in model.esm_ln.parameters():
        p.requires_grad = False
    for p in model.drfp_ln.parameters():
        p.requires_grad = False
    for p in model.react_ln.parameters():
        p.requires_grad = False
    for p in model.compressor.parameters():
        p.requires_grad = False
    print("Frozen normalization layers and compressor. Only KAN head will be fine-tuned.")


In [ ]:
# @title 10. Fine-tune

y_tr = torch.tensor(df_tr[LABEL_COL].to_numpy(np.float32))
y_va = torch.tensor(df_va[LABEL_COL].to_numpy(np.float32))

n_pos = int((df_tr[LABEL_COL] == 1).sum())
n_neg = int((df_tr[LABEL_COL] == 0).sum())

if POS_WEIGHT == "auto":
    pos_weight_val = n_neg / max(1, n_pos)
else:
    pos_weight_val = float(POS_WEIGHT)

print(f"Train positives={n_pos}, negatives={n_neg}, pos_weight={pos_weight_val:.4f}")

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_val], dtype=torch.float32, device=device)
)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

best_val = -1e9
best_epoch = -1
bad_epochs = 0
train_logs = []

best_path = f"checkpoints/ezhit_finetuned_seed{SEED}.pt"

for ep in range(1, EPOCHS + 1):
    model.train()
    perm = np.random.permutation(len(df_tr))
    losses = []

    for i in tqdm(range(0, len(perm), BATCH_SIZE), desc=f"Epoch {ep}/{EPOCHS}"):
        b = perm[i:i+BATCH_SIZE]
        e_b = esm_tr[b].to(device)
        f_b = fp_tr[b].to(device)
        r_b = react_tr[b].to(device)
        y_b = y_tr[b].to(device)

        logit = model(e_b, f_b, r_b)
        loss = criterion(logit, y_b)

        if hasattr(model.net, "regularization_loss"):
            loss = loss + KAN_REG_WEIGHT * model.net.regularization_loss()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        losses.append(float(loss.detach().cpu().item()))

    val_scores = predict_scores_and_latents(model, esm_va, fp_va, react_va, batch_size=4096, return_latents=False)
    val_metrics = evaluate_groups_ranking(df_va, val_scores, GROUP_COL, LABEL_COL)

    sel = float(val_metrics[SELECT_METRIC])
    improved = (not np.isnan(sel)) and (sel > best_val + MIN_DELTA)

    row = {
        "epoch": ep,
        "train_loss": float(np.mean(losses)),
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "selected_metric": SELECT_METRIC,
        "selected_value": sel,
        "improved": improved
    }
    train_logs.append(row)

    print(
        f"Epoch {ep:03d} | loss={row['train_loss']:.4f} | "
        f"val_top1={val_metrics['top1']:.4f} val_top10={val_metrics['top10']:.4f} "
        f"val_dcg@10={val_metrics['dcg@10']:.4f} val_auc={val_metrics['auc_micro']:.4f} | "
        f"select={SELECT_METRIC}:{sel:.4f} best={best_val:.4f} improved={improved}"
    )

    if improved:
        best_val = sel
        best_epoch = ep
        bad_epochs = 0
        torch.save({
            "epoch": best_epoch,
            "model": model.state_dict(),
            "args": {
                "hidden": hidden_dim,
                "esm_dim": int(esm_tr.shape[1]),
                "drfp_dim": int(fp_tr.shape[1]),
                "react_dim": int(react_tr.shape[1]),
                "dropout": DROPOUT,
                "seq_col": SEQ_COL,
                "rxn_col": RXN_COL,
                "label_col": LABEL_COL,
                "group_col": GROUP_COL,
                "source_checkpoint": os.path.basename(PRETRAINED_CKPT),
                "seed": SEED,
            },
            "val_metrics": val_metrics,
        }, best_path)
        print(f"Saved best checkpoint -> {best_path}")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {ep}.")
            break

pd.DataFrame(train_logs).to_csv("outputs/training_log.csv", index=False)
print(f"Best epoch={best_epoch}, best {SELECT_METRIC}={best_val:.4f}")


In [ ]:
# @title 11. Evaluate on validation/test and save predictions

best_ckpt = torch.load(best_path, map_location="cpu")
model.load_state_dict(best_ckpt["model"])
model.to(device).eval()

def save_predictions(df_eval, esm_eval, fp_eval, react_eval, split_name):
    scores = predict_scores_and_latents(
        model,
        esm_eval,
        fp_eval,
        react_eval,
        batch_size=4096,
        return_latents=False
    )
    out = df_eval.copy()
    out["EZHit_probability"] = scores
    out.to_csv(f"outputs/{split_name}_predictions.csv", index=False)

    metrics = evaluate_groups_ranking(out, scores, GROUP_COL, LABEL_COL)
    ef = calculate_local_ef(out, scores, GROUP_COL, LABEL_COL)

    print("\n" + "="*60)
    print(split_name.upper(), "METRICS")
    print("="*60)
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
    for k, v in ef.items():
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

    return out, metrics, ef

val_pred, val_metrics, val_ef = save_predictions(df_va, esm_va, fp_va, react_va, "val")

if len(df_te) > 0 and esm_te is not None:
    test_pred, test_metrics, test_ef = save_predictions(df_te, esm_te, fp_te, react_te, "test")
else:
    print("No test split available.")


In [ ]:
# @title 12. Generate train_distribution_stat.pt for Mahalanobis distance

# This statistic is computed from positive training pairs after fine-tuning.
# Upload this file to your HuggingFace Space together with the fine-tuned checkpoint
# if you want Space inference to report Mahalanobis distance.

@torch.no_grad()
def extract_latents(model, esm, fp, react, batch_size=4096):
    _, latents = predict_scores_and_latents(
        model, esm, fp, react,
        batch_size=batch_size,
        return_latents=True
    )
    return latents.float()

pos_mask = df_tr[LABEL_COL].to_numpy().astype(int) == 1
if pos_mask.sum() < 2:
    raise ValueError("Need at least two positive training pairs to compute covariance.")

pos_indices = np.where(pos_mask)[0]
z_pos = extract_latents(
    model,
    esm_tr[pos_indices],
    fp_tr[pos_indices],
    react_tr[pos_indices],
    batch_size=4096
)

mu = z_pos.mean(dim=0)

# Shrunk covariance for numerical stability.
z_centered = z_pos - mu
n = z_centered.shape[0]
cov = (z_centered.T @ z_centered) / max(1, n - 1)
cov = cov + MAHA_EPS * torch.eye(cov.shape[0])

inv_cov = torch.linalg.pinv(cov)

stat = {
    "mean": mu.cpu(),
    "inv_cov": inv_cov.cpu(),
    "n_positive": int(pos_mask.sum()),
    "hidden_dim": int(mu.numel()),
    "source_checkpoint": os.path.basename(best_path),
    "note": "Computed from positive fine-tuning training pairs."
}

torch.save(stat, "outputs/train_distribution_stat.pt")

print("Saved Mahalanobis statistics -> outputs/train_distribution_stat.pt")
print("mean:", tuple(mu.shape), "inv_cov:", tuple(inv_cov.shape), "n_positive:", int(pos_mask.sum()))


In [ ]:
# @title 13. Package and download results

import zipfile
from google.colab import files

zip_path = "EZHit_finetuning_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ["checkpoints", "outputs", "cache"]:
        for root, _, file_names in os.walk(folder):
            for fn in file_names:
                path = os.path.join(root, fn)
                z.write(path, arcname=path)

print("Created:", zip_path)
files.download(zip_path)
